# Customer Intelligence Analytics with spaCy & Hugging Face

- 🏆 100 points available
- 🤠 Author: Park

## 💎 Case Overview

You are a Data Scientist at a tech-forward media tracking company. Your team intercepts thousands of news articles and public forum posts daily. Your objective is to build an automated pipeline that extracts key entities (like company names) using rule-based NLP, cleans the text, and then routes the text through advanced Deep Learning models to determine sentiment and categorize the topic—all without training a model from scratch.

### 🐢 Your Dataset

- We will use a small, real-world slice of the `ag_news` (AG's News Corpus) dataset via the Hugging Face `datasets` library.

### ⚔️ Your Goal

Analyze the dataset and complete the following NLP tasks:

- 👉 Extract unique Organizations (ORG) using `spaCy`.
- 👉 Tokenize, lemmatize, and clean text using `spaCy`.
- 👉 Determine text sentiment using a pre-trained Hugging Face transformer.
- 👉 Categorize text using a Zero-Shot Classification Hugging Face model.
- 👉 Apply these functions to your DataFrame.


---

▶️ Run the code cell below to import `unittest`, a module used for **🧭 Check Your Work** sections and the autograder.


In [3]:
# DO NOT MODIFY THE CODE IN THIS CELL
import base64
import unittest

tc = unittest.TestCase()

---

### 🔨 Import Packages and Dataset

▶️ Run the code cell below to install and import packages, and to load the dataset.
_(Note: You may need to uncomment the `!pip install` and `!python -m spacy download` lines if running locally)_

**PyTorch Installation Note**

If you're running this notebook locally, you will also need PyTorch, a deep learning framework, to run the Hugging Face models. Please refer to the [PyTorch installation guide](https://pytorch.org/get-started/locally/) to install the appropriate version for your system.


In [4]:
# !pip install spacy transformers datasets pandas
# !python -m spacy download en_core_web_sm

import pandas as pd
import spacy
from transformers import pipeline
from datasets import load_dataset

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

# Load a slice of the ag_news dataset
print("Loading dataset...")
dataset = load_dataset("ag_news", split="train[:20]")
df = pd.DataFrame(dataset)

# Insert custom test records
custom_texts = pd.DataFrame(
    {
        "text": [
            "Apple Inc. reported a massive profit for the third quarter. The CEO, Tim Cook, was thrilled with the sales of the new iPhone.",
            "The new software update from Microsoft is terrible and completely broke my computer system!",
        ],
        "label": [3, 4],
    }
)
df = pd.concat([custom_texts, df], ignore_index=True)
print(f"Dataset loaded with {len(df)} records.")
df.head()

OSError: [WinError 127] The specified procedure could not be found. Error loading "c:\Users\ypark32\AppData\Local\miniforge3\Lib\site-packages\torch\lib\shm.dll" or one of its dependencies.

---

## 📐 Part 1: Rule-Based Preprocessing


---

### 🎯 Deliverable 1: Extract Organizations using spaCy

#### 👇 Tasks

- ✔️ Create a function named `extract_organizations()`.
- ✔️ The function should take a string `text` as input.
- ✔️ Parse the `text` using `nlp` (the spaCy model).
- ✔️ Return a list of all unique entities labeled as `'ORG'`.
- The return should be a `set` of unique organization names.

#### 🚀 Hint

You can iterate through `doc.ents` and check `ent.label_`. Use `set()` to create a new `set` data structure that automatically handles uniqueness.


In [ ]:
def extract_organizations(text):
    # YOUR CODE BEGINS
    doc = nlp(text)
    orgs = set()
    for ent in doc.ents:
        if ent.label_ == "ORG":
            orgs.add(ent.text)
    return orgs
    # YOUR CODE ENDS

#### 🧭 Check Your Work

Run the code cell below to test your solution.

- ✔️ If the code cell runs without errors, you're good to move on.
- ❌ If the code cell produces an error, review your code and fix any mistakes.


In [ ]:
_test_case = "deliverable-01"
_points = 20

sample_text = (
    "Google and Microsoft are competing in the AI space. Google is investing heavily."
)
orgs = extract_organizations(sample_text)

tc.assertTrue("Google" in orgs, "Failed: 'Google' not found.")
tc.assertTrue("Microsoft" in orgs, "Failed: 'Microsoft' not found.")
tc.assertEqual(len(orgs), 2, "Failed: Should only return unique organizations.")
print("Task 1 (NER) Passed!")

---

### 🎯 Deliverable 2: Clean and Lemmatize using spaCy

#### 👇 Tasks

- ✔️ Create a function named `clean_and_lemmatize()`.
- ✔️ The function should take a string `text` as input.
- ✔️ Parse the text using `nlp`.
- ✔️ Iterate through the tokens, and filter out any token where `token.is_stop`, `token.is_punct`, or `token.is_space` is True.
- ✔️ Lowercase the remaining tokens and lemmatize them using `token.lemma_.lower()`.
- ✔️ Return a list of cleaned and lemmatized tokens.


In [ ]:
# YOUR CODE BEGINS
def clean_and_lemmatize(text):

    doc = nlp(text)
    lemmas = [
        token.lemma_.lower()
        for token in doc
        if not token.is_stop and not token.is_punct and not token.is_space
    ]
    return lemmas


# YOUR CODE ENDS

print(
    clean_and_lemmatize(
        "Natural language processing is amazing with new models available on HuggingFace!"
    )
)

#### 🧭 Check Your Work

Run the code cell below to test your solution.

- ✔️ If the code cell runs without errors, you're good to move on.
- ❌ If the code cell produces an error, review your code and fix any mistakes.


In [ ]:
_test_case = "deliverable-02"
_points = 20

sample_text = "The quick brown foxes are jumping over the lazy dogs!"
cleaned = clean_and_lemmatize(sample_text)
tc.assertIn("fox", cleaned, "Failed: 'foxes' was not lemmatized to 'fox'.")
tc.assertIn("dog", cleaned, "Failed: 'dogs' was not lemmatized to 'dog'.")
tc.assertNotIn("the", cleaned, "Failed: Stopwords were not removed.")
tc.assertNotIn("!", cleaned, "Failed: Punctuation was not removed.")
print("Task 2 (Preprocessing) Passed!")

---

## 🤖 Part 2: Deep Learning with Hugging Face


---

### 🎯 Deliverable 3: Sentiment Analysis Model

#### 👇 Tasks

- ✔️ Instantiate a `pipeline` for `"sentiment-analysis"` using the model `"distilbert-base-uncased-finetuned-sst-2-english"` and store it in `sentiment_pipeline`.
- ✔️ Complete the function `get_sentiment(text)` to return the label (`'POSITIVE'` or `'NEGATIVE'`) from the pipeline's output.


In [ ]:
print("Loading Sentiment Analysis Model...")
# YOUR CODE BEGINS
sentiment_pipeline = pipeline(
    "sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english"
)


def get_sentiment(text):
    result = sentiment_pipeline(text)
    return result[0]["label"]


# YOUR CODE ENDS

#### 🧭 Check Your Work

Run the code cell below to test your solution.

- ✔️ If the code cell runs without errors, you're good to move on.
- ❌ If the code cell produces an error, review your code and fix any mistakes.


In [ ]:
_test_case = "deliverable-03"
_points = 20

pos_text = "I absolutely love the new design, it is fantastic!"
neg_text = "This is the worst experience I have ever had."
tc.assertEqual(
    get_sentiment(pos_text), "POSITIVE", "Failed: Did not identify positive sentiment."
)
tc.assertEqual(
    get_sentiment(neg_text), "NEGATIVE", "Failed: Did not identify negative sentiment."
)
print("Task 3 (Sentiment Analysis) Passed!")

---

### 🎯 Deliverable 4: Zero-Shot Classification Model

#### 👇 Tasks

- ✔️ Instantiate a `pipeline` for `"zero-shot-classification"` using the model `"facebook/bart-large-mnli"` and store it in `zero_shot_pipeline`.
- ✔️ Complete the function `categorize_topic(text, candidate_labels)` to return the most likely label (the first one in the list of returned labels).


In [ ]:
print("Loading Zero-Shot Classification Model...")
# YOUR CODE BEGINS
zero_shot_pipeline = pipeline(
    "zero-shot-classification", model="facebook/bart-large-mnli"
)


def categorize_topic(text, candidate_labels):
    result = zero_shot_pipeline(text, candidate_labels)
    return result["labels"][0]


# YOUR CODE ENDS

#### 🧭 Check Your Work

Run the code cell below to test your solution.

- ✔️ If the code cell runs without errors, you're good to move on.
- ❌ If the code cell produces an error, review your code and fix any mistakes.


In [ ]:
_test_case = "deliverable-04"
_points = 20

sample_text = "Apple Inc. released a new update for the iPhone battery life."
labels = ["Technology", "Politics", "Sports"]
top_label = categorize_topic(sample_text, labels)
tc.assertEqual(
    top_label, "Technology", f"Failed: Expected 'Technology', got {top_label}"
)
print("Task 4 (Zero-Shot Classification) Passed!")

---

## 🚀 Part 3: Execution


---

### 🎯 Deliverable 5: Apply Pipelines to DataFrame

#### 👇 Tasks

- ✔️ Use the `.apply()` method on the `text` column of `df` to extract organizations and store the result in a new column called `extracted_orgs`.
- ✔️ Use the `.apply()` method on the `text` column of `df` to get sentiments and store the result in a new column called `sentiment`.


In [ ]:
# YOUR CODE BEGINS
df["extracted_orgs"] = df["text"].apply(extract_organizations)
df["sentiment"] = df["text"].apply(get_sentiment)
# YOUR CODE ENDS
display(df[["text", "extracted_orgs", "sentiment"]].head(5))

#### 🧭 Check Your Work

Run the code cell below to test your solution.

- ✔️ If the code cell runs without errors, you're good to move on.
- ❌ If the code cell produces an error, review your code and fix any mistakes.


In [ ]:
_test_case = "deliverable-05"
_points = 20

tc.assertIn("extracted_orgs", df.columns, "Failed: 'extracted_orgs' column not found.")
tc.assertIn("sentiment", df.columns, "Failed: 'sentiment' column not found.")
tc.assertEqual(
    df.loc[0, "sentiment"],
    "POSITIVE",
    "Failed: Sentiment for first row should be POSITIVE.",
)
tc.assertEqual(
    df.loc[1, "sentiment"],
    "NEGATIVE",
    "Failed: Sentiment for second row should be NEGATIVE.",
)
print("Task 5 (Applying Pipelines) Passed!")